# KHILONA Color Classifier

Deep learning classifier for blue, yellow, purple objects. No pretrained models. Provide a dataset directory with three subfolders: `blue`, `yellow`, `purple` containing images.

In [ ]:
import os, json, math, random
import numpy as np
from pathlib import Path
from PIL import Image
np.random.seed(42)

In [ ]:
data_dir = "datasets/khilona_team"
img_size = 32
hidden = 128
epochs = 60
batch_size = 64
lr = 5e-3
val_split = 0.15
test_split = 0.15
classes = ["blue","yellow","purple"]
belts = ["A","B","C"]
out_model = "khilona_model.npz"

In [ ]:
def load_images(root, classes, img_size):
    X = []
    y = []
    paths = []
    for idx, c in enumerate(classes):
        cls_dir = Path(root) / c
        for p in cls_dir.glob("*.*"):
            try:
                img = Image.open(p).convert("RGB").resize((img_size,img_size))
                arr = np.asarray(img, dtype=np.float32)/255.0
                X.append(arr.transpose(2,0,1).reshape(-1))
                y.append(idx)
                paths.append(str(p))
            except:
                pass
    X = np.stack(X,0)
    y = np.array(y, dtype=np.int64)
    return X, y, paths

X, y, paths = load_images(data_dir, classes, img_size)
n = X.shape[0]
idxs = np.arange(n)
np.random.shuffle(idxs)
n_test = int(n*test_split)
n_val = int(n*val_split)
n_train = n - n_val - n_test
train_idx = idxs[:n_train]
val_idx = idxs[n_train:n_train+n_val]
test_idx = idxs[n_train+n_val:]
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

In [ ]:
mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-6
def norm(a):
    return (a - mu)/sigma
X_train_n = norm(X_train)
X_val_n = norm(X_val)
X_test_n = norm(X_test)

In [ ]:
class MLP:
    def __init__(self, in_dim, hidden, out_dim, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0,0.05,(in_dim,hidden)).astype(np.float32)
        self.b1 = np.zeros((hidden,), np.float32)
        self.W2 = rng.normal(0,0.05,(hidden,out_dim)).astype(np.float32)
        self.b2 = np.zeros((out_dim,), np.float32)
        self.mW1 = np.zeros_like(self.W1); self.vW1 = np.zeros_like(self.W1)
        self.mb1 = np.zeros_like(self.b1); self.vb1 = np.zeros_like(self.b1)
        self.mW2 = np.zeros_like(self.W2); self.vW2 = np.zeros_like(self.W2)
        self.mb2 = np.zeros_like(self.b2); self.vb2 = np.zeros_like(self.b2)
        self.t = 0
    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = np.maximum(z1,0)
        z2 = a1 @ self.W2 + self.b2
        e = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
        y = e / np.sum(e, axis=1, keepdims=True)
        return z1, a1, y
    def loss_acc(self, X, y_true):
        _, _, y = self.forward(X)
        n = X.shape[0]
        loss = -np.mean(np.log(y[np.arange(n), y_true] + 1e-8))
        pred = np.argmax(y, axis=1)
        acc = np.mean(pred == y_true)
        return loss, acc
    def backward(self, X, y_true, z1, a1, y):
        n = X.shape[0]
        gy = y.copy()
        gy[np.arange(n), y_true] -= 1.0
        gy /= n
        gW2 = a1.T @ gy
        gb2 = gy.sum(axis=0)
        ga1 = gy @ self.W2.T
        gz1 = ga1 * (z1 > 0)
        gW1 = X.T @ gz1
        gb1 = gz1.sum(axis=0)
        return gW1, gb1, gW2, gb2
    def step_adam(self, grads, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        gW1, gb1, gW2, gb2 = grads
        for name, g, param in [("W1", gW1, self.W1), ("b1", gb1, self.b1), ("W2", gW2, self.W2), ("b2", gb2, self.b2)]:
            m = getattr(self,"m"+name); v = getattr(self,"v"+name)
            m[:] = b1*m + (1-b1)*g
            v[:] = b2*v + (1-b2)*(g*g)
            mh = m/(1-b1**self.t)
            vh = v/(1-b2**self.t)
            param[:] = param - lr*mh/(np.sqrt(vh)+eps)

In [ ]:
model = MLP(X_train_n.shape[1], hidden, len(classes), seed=123)
order = np.arange(X_train_n.shape[0])
for ep in range(1, epochs+1):
    np.random.shuffle(order)
    for i in range(0, len(order), batch_size):
        b = order[i:i+batch_size]
        xb = X_train_n[b]; yb = y_train[b]
        z1, a1, yhat = model.forward(xb)
        grads = model.backward(xb, yb, z1, a1, yhat)
        model.step_adam(grads, lr=lr)
    tr_loss, tr_acc = model.loss_acc(X_train_n, y_train)
    va_loss, va_acc = model.loss_acc(X_val_n, y_val)
    print(ep, round(tr_loss,4), round(tr_acc,4), round(va_loss,4), round(va_acc,4))

In [ ]:
te_loss, te_acc = model.loss_acc(X_test_n, y_test)
print("test_acc", round(te_acc,4))
np.savez(out_model, W1=model.W1, b1=model.b1, W2=model.W2, b2=model.b2, mu=mu, sigma=sigma, classes=np.array(classes), img_size=np.array([img_size]))

In [ ]:
def predict_image(image_path, model_path=out_model):
    d = np.load(model_path, allow_pickle=True)
    W1, b1, W2, b2 = d["W1"], d["b1"], d["W2"], d["b2"]
    mu, sigma = d["mu"], d["sigma"]
    cls = d["classes"].tolist()
    img_size_loaded = int(d["img_size"][0])
    img = Image.open(image_path).convert("RGB").resize((img_size_loaded, img_size_loaded))
    x = np.asarray(img, dtype=np.float32)/255.0
    x = x.transpose(2,0,1).reshape(-1)[None, ...]
    x = (x - mu)/sigma
    z1 = x @ W1 + b1
    a1 = np.maximum(z1,0)
    z2 = a1 @ W2 + b2
    e = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
    y = e / np.sum(e, axis=1, keepdims=True)
    idx = int(np.argmax(y, axis=1)[0])
    return cls[idx], float(y[0, idx])

pred_class, prob = predict_image("path/to/object.jpg")
belt = belts[classes.index(pred_class)]
print(pred_class, belt, round(prob,4))

In [ ]:
def majority_color(image_path, target_size=img_size):
    img = Image.open(image_path).convert("RGB").resize((target_size, target_size))
    hsv = img.convert("HSV")
    h, s, v = [np.asarray(ch, dtype=np.float32) for ch in hsv.split()]
    mask = (s > 40) & (v > 40)
    hdeg = h * (360.0 / 255.0)
    blue_mask = (hdeg >= 190) & (hdeg <= 260)
    yellow_mask = (hdeg >= 40) & (hdeg <= 70)
    purple_mask = (hdeg >= 270) & (hdeg <= 320)
    counts = [int(np.sum(blue_mask & mask)), int(np.sum(yellow_mask & mask)), int(np.sum(purple_mask & mask))]
    if np.sum(mask) == 0:
        idx = 0
        share = 0.0
    else:
        idx = int(np.argmax(counts))
        share = float(counts[idx]) / float(np.sum(mask))
    return classes[idx], share, counts

def predict_and_display(image_path, mode="dl"):
    if mode == "majority":
        c, share, _ = majority_color(image_path)
        p = share
    else:
        c, p = predict_image(image_path)
    belt = belts[classes.index(c)]
    print("class=", c, "belt=", belt, "confidence=", round(p,4))
    img = Image.open(image_path).convert("RGB")
    display(img)

# Example:
# predict_and_display("path/to/object.jpg", mode="dl")
# predict_and_display("path/to/object.jpg", mode="majority")

In [ ]:
try:
    import cv2
    def run_camera(mode="dl", cam_index=0):
        cap = cv2.VideoCapture(cam_index)
        if not cap.isOpened():
            print("camera not available")
            return
        d = None
        if mode == "dl":
            d = np.load(out_model, allow_pickle=True)
            W1, b1, W2, b2 = d["W1"], d["b1"], d["W2"], d["b2"]
            mu, sigma = d["mu"], d["sigma"]
            img_s = int(d["img_size"][0])
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if mode == "majority":
                # Save to temp buffer for PIL
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(rgb)
                img = img.resize((img_size, img_size))
                img.save("_tmp_cam.jpg")
                c, p, _ = majority_color("_tmp_cam.jpg", target_size=img_size)
            else:
                # DL path
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(rgb).resize((img_s, img_s))
                x = (np.asarray(img, dtype=np.float32)/255.0).transpose(2,0,1).reshape(-1)[None,...]
                x = (x - mu)/sigma
                z1 = x @ W1 + b1
                a1 = np.maximum(z1,0)
                z2 = a1 @ W2 + b2
                e = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
                y = e / np.sum(e, axis=1, keepdims=True)
                idx = int(np.argmax(y, axis=1)[0])
                c = classes[idx]
                p = float(y[0, idx])
            belt = belts[classes.index(c)]
            txt = f"{c.upper()} | Belt {belt} | {p:.2f}"
            cv2.putText(frame, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2, cv2.LINE_AA)
            cv2.imshow("KHILONA Camera", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()
except Exception as e:
    print("cv2 not available; install opencv-python to enable camera.")

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    mode = widgets.ToggleButtons(options=["dl","majority"], description="mode")
    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    out = widgets.Output()
    def on_click(b=None):
        out.clear_output()
        if len(uploader.value) == 0:
            return
        info = next(iter(uploader.value.values()))
        content = info["content"]
        tmp = "_upload_tmp.jpg"
        with open(tmp, "wb") as f:
            f.write(content)
        with out:
            predict_and_display(tmp, mode.value)
    btn = widgets.Button(description="Predict")
    btn.on_click(on_click)
    display(mode, uploader, btn, out)
except Exception as e:
    print("ipywidgets not available")